# 🌱 Walk & Notice

### Hacktoberfest 2026 — Touch Grass

> **AI should help you leave the screen, not keep you on it.**

**Walk & Notice** is an open-source AI-powered outdoor observation
companion that generates short, personalized missions designed to
encourage people to step outside, observe their surroundings, and
temporarily put their phone away.

### Challenge
Hacktoberfest 2026 — Week 1: **Touch Grass**

### Core Technologies
- Python
- Open-weight LLM
- OpenRouter / Ollama
- Streamlit
- Local-first AI architecture

---

### Product Principle

**Generate → Go Outside → Observe → Return → Reflect**

The AI is intentionally designed to become unnecessary once the
outdoor mission begins.

## 🌍 The Problem

Modern digital experiences are designed to keep users on screens.

Even applications intended to promote wellness, exploration, or
outdoor activity can increase screen time.

### The question

**Can AI be used to help people disconnect from technology rather
than spend more time interacting with it?**

Walk & Notice explores this idea by using AI for only one short task:

> **Create a personalized outdoor observation mission.**

After that, the user is encouraged to put the phone away and experience
the real world.

## 🏗️ Prototype Architecture

```text
User Preferences
       │
       ▼
Streamlit Interface
       │
       ▼
Prompt Builder
       │
       ▼
Open-Weight LLM
       │
   ┌───┴────┐
   │        │
OpenRouter Ollama
   │        │
   └───┬────┘
       ▼
Mission Validation
       │
       ▼
Outdoor Mission
       │
       ▼
📱 PUT PHONE AWAY
       │
       ▼
🌳 REAL-WORLD EXPERIENCE

In [4]:
## Cell 5 — Setup

# ============================================
# WALK & NOTICE — Prototype Setup
# ============================================

!pip -q install openai python-dotenv gradio

from pathlib import Path
import os
import json
import re
import random
from getpass import getpass

PROJECT_DIR = Path("/content/walk-and-notice")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Environment ready")

✅ Environment ready


In [6]:
# ============================================
# OpenRouter Configuration
# ============================================

from google.colab import userdata
from openai import OpenAI

# Fetch API key securely from Google Colab Secrets
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "❌ OPENROUTER_API_KEY not found in Colab Secrets. "
        "Add it under the 🔑 Secrets panel."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "openai/gpt-oss-20b:free"

print("✅ OpenRouter configured successfully")
print(f"🤖 Model: {MODEL}")

✅ OpenRouter configured successfully
🤖 Model: openai/gpt-oss-20b:free


In [9]:
# ============================================
# OpenRouter Diagnostic Test
# ============================================

MODEL = "openai/gpt-oss-20b"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Say exactly: Walk & Notice is ready."
        }
    ],
    temperature=0.2,
    max_tokens=50,
)

print("Response object received:")
print(response)

print("\n--- Choices ---")
print(response.choices)

if response.choices:
    print("\n--- Message ---")
    print(response.choices[0].message)

    print("\n--- Content ---")
    print(repr(response.choices[0].message.content))

Response object received:
ChatCompletion(id='gen-1791551056-i1wLJY9V9ebrufwkWQWV', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Walk & Notice is ready.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user says: "Say exactly: Walk & Notice is ready." They want the assistant to say exactly that phrase. So I should respond with exactly that phrase. No extra punctuation? The phrase is "Walk & Notice is ready." So I should output that. Probably no quotes. Just that line.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user says: "Say exactly: Walk & Notice is ready." They want the assistant to say exactly that phrase. So I should respond with exactly that phrase. No extra punctuation? The phrase is "Walk & Notice is ready." So I should output that. Probably no quotes. Just that line.', 'format': 'unknown', 'index': 0}]), native_finish_reason='stop')], crea

## 🤖 AI Mission Generator

The core intelligence of Walk & Notice is a structured mission
generator.

Instead of asking the model for a generic outdoor activity, we provide
specific constraints:

- Duration
- Environment
- Interest
- Energy level

The model must return a concise mission that can be completed without
continuously interacting with a screen.

The final instruction deliberately encourages the user to put the
phone away.

In [22]:
# ============================================
# Walk & Notice — Compact Mission Prompt
# ============================================

MISSION_SYSTEM_PROMPT = """
You are Walk & Notice, an outdoor observation mission generator.

Generate ONE safe outdoor mission.

STRICT OUTPUT FORMAT:
1. Write exactly 4 numbered steps.
2. Each step must be ONE short sentence.
3. Maximum 12 words per step.
4. After step 4, write exactly:
Now put your phone away.

SAFETY:
- Stay on public paths.
- Do not approach wildlife.
- Do not enter traffic or unsafe areas.
- Do not climb or touch dangerous objects.
- Do not require equipment.
- Do not require a phone, camera, or recording.

The mission must match the user's duration, environment,
interest, and energy level.

Return ONLY the mission.
"""

In [23]:
# ============================================
# Mission Generation Test
# ============================================

mission = generate_mission(
    duration=15,
    environment="urban park",
    interest="birds",
    energy="low",
)

print(mission)

1. Walk to the park's shaded pond.  
2. Sit on the bench beside the pond.  
3. Observe sparrows and finches for 10 minutes.  
4. Note any bird songs in your mind.  
Now put your phone away.
